# Opentrons OT-2

The OT-2 is a two-mount liquid-handling robot. PyLabRobot discovers the pipette on each mount and exposes it as a real object, so operations read as `pipette.pick_up_tip(...)`, `pipette.aspirate(...)`, and `pipette.dispense(...)`.

| Property | Value |
|---|---|
| Communication | JSON over HTTP |
| Default address | Robot hostname or IP, port `31950` |
| Pipette mounts | Left and right |
| Supported liquid operations | Single-channel GEN1 and GEN2 pipettes |
| Deck | 12 slots; slot 12 contains fixed trash by default |

```{warning}
This new-architecture driver has NOT been tested against hardware in PyLabRobot. `setup()` logs a warning to that effect. Keep clear of the deck whenever the robot can move. If you verify it on your OT-2, please open a PR to remove the warning.
```

The OT-2 exposes a run-command HTTP API. PyLabRobot creates a run during `setup()`, sends one command at a time, and waits for each command to succeed or fail before continuing.

## Physical setup

1. Install the pipettes and remove any tips already attached to their nozzles.
2. In the Opentrons App, complete deck calibration, pipette-offset calibration, and tip-length calibration for the exact Opentrons tip rack you will use.
3. Put the computer and OT-2 on the same network.
4. Find the robot's hostname or IP in the Opentrons App. A hostname such as `ot2.local` may also work on your network.
5. Keep the deck clear until the labware layout below matches the physical deck.

## Create the robot

Create the deck first and pass it to the robot. Replace `ot2.local` with your robot's hostname or IP address, without `http://`.

In [ ]:
from pylabrobot.opentrons import OpentronsOT2
from pylabrobot.resources import OTDeck

deck = OTDeck()
ot2 = OpentronsOT2(host="ot2.local", deck=deck)

## Connect

`setup()` creates an Opentrons run, discovers the mounted pipettes, reads the robot API version, and homes the robot.

In [ ]:
await ot2.setup()

## Inspect the pipettes

The left and right mount are either an `OT2Pipette` or `None`. This notebook uses the first mounted single-channel pipette.

In [ ]:
print("Left:", ot2.left_pipette.name if ot2.left_pipette else None)
print("Right:", ot2.right_pipette.name if ot2.right_pipette else None)

pipette = next((p for p in ot2.pipettes if p.channels == 1), None)
assert pipette is not None, "This example needs a mounted single-channel pipette"

## Model the physical deck

Choose a tip rack that exactly matches the physical rack and discovered pipette, place it in slot 1, and place the plate in slot 2. Make the physical deck match this layout before continuing. The standard Opentrons rack definitions below preserve the rack identity used by the robot's tip-length calibration. Tracking is enabled so PyLabRobot checks tip and liquid state around each robot command.

In [ ]:
from pylabrobot.resources import set_tip_tracking, set_volume_tracking
from pylabrobot.resources.celltreat import celltreat_96_wellplate_350uL_Fb
from pylabrobot.resources.opentrons import (
  opentrons_96_filtertiprack_10ul,
  opentrons_96_filtertiprack_20ul,
  opentrons_96_filtertiprack_200ul,
  opentrons_96_filtertiprack_1000ul,
  opentrons_96_tiprack_300ul,
)

set_tip_tracking(True)
set_volume_tracking(True)

tip_rack_factory = {
  10: opentrons_96_filtertiprack_10ul,
  20: opentrons_96_filtertiprack_20ul,
  50: opentrons_96_filtertiprack_200ul,
  300: opentrons_96_tiprack_300ul,
  1000: opentrons_96_filtertiprack_1000ul,
}[pipette.maximum_volume]

tips = tip_rack_factory(name="tips")
plate = celltreat_96_wellplate_350uL_Fb(name="plate")
deck.assign_child_at_slot(tips, slot=1)
deck.assign_child_at_slot(plate, slot=2)

transfer_volume = max(pipette.minimum_volume, min(20, pipette.maximum_volume))
test_liquid_volume = max(100, transfer_volume * 2)
plate.get_well("A1").tracker.set_volume(test_liquid_volume)
print(f"Before continuing, manually add {test_liquid_volume:g} µL of water to plate well A1.")

## Pick up a tip

Pause here and add the printed amount of water to plate well A1. Verify that the matching tip rack is physically in slot 1, the plate is in slot 2, and tip A1 is present. This is the first operation after homing that approaches labware.

In [ ]:
await pipette.pick_up_tip(tips.get_item("A1"))

## Mix

`mix()` moves to 1 mm above the well bottom, performs the requested aspiration/dispense cycles client-side, then returns to traversal height.

In [ ]:
await pipette.mix(
  plate.get_well("A1"), volume=transfer_volume, repetitions=3, liquid_height=1
)

## Aspirate

Aspirate from the cavity bottom plus `liquid_height`. You can also pass a `Coordinate` offset to compensate for a carefully measured positional calibration difference.

In [ ]:
await pipette.aspirate(plate.get_well("A1"), volume=transfer_volume, liquid_height=1)

## Dispense

Dispense the tracked liquid into another well. The pipette returns to the configured traversal height after the operation.

In [ ]:
await pipette.dispense(plate.get_well("B1"), volume=transfer_volume, liquid_height=1)

## Return the tip

`return_tip()` uses the recorded pickup origin and restores the tip-rack tracker after the robot command succeeds.

In [ ]:
await pipette.return_tip()

## Pick up another tip

Pick up a fresh tip to demonstrate disposal in the fixed trash.

In [ ]:
await pipette.pick_up_tip(tips.get_item("A2"))

## Discard the tip

`discard_tip()` uses the fixed-trash command sequence appropriate for the robot's reported HTTP API version.

In [ ]:
await pipette.discard_tip()

## Home

Home the gantry and pipette axes when you need to return the robot to its reference state.

In [ ]:
await ot2.home()

## Teardown

Always run `stop()`, including after an error. It stops the active Opentrons run, making the robot available to the Opentrons App again, and closes the HTTP transport.

In [ ]:
await ot2.stop()